# 01. Evaluate a shipped agent

Loads `UECD-SingleMap-Best` (the dissertation's headline agent) and plays
two games against the field:

1. One game vs `RandomBiasedAI` (weakest scripted bot).
2. One game vs `CoacAI` (strong competition bot).

Two ways to run inference are shown:

- **CLI**: `microrts-agent evaluate ...` (recommended, what every script does).
- **Python**: `load_agent_from_config(...)` to inspect / use the model directly.

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.

## 1. Load the agent in Python and inspect its config

Every shipped agent under `data/agents/<name>/` is rebuildable from its
`config.json` + `agent.pt`. Helper `load_agent_from_config` reads the
config, instantiates the matching architecture, and loads the weights.

In [ ]:
from microrts_agent.architectures.factory import load_agent_from_config
from microrts_agent.paths import PROJECT_ROOT

agent_dir = PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"
agent, config = load_agent_from_config(str(agent_dir), device="cpu")
agent.eval()

interesting_keys = [
    "architecture",
    "obs_channels",
    "action_nvec",
    "extended_obs",
    "filtered_masks",
    "reserved_obs",
    "total_timesteps",
    "reward_weight",
]
for k in interesting_keys:
    if k in config:
        print(f"  {k:18s} = {config[k]}")

n_params = sum(p.numel() for p in agent.parameters())
print(f"\nTotal parameters: {n_params:,}")

## 2. Play 1 game vs `RandomBiasedAI`

The CLI wraps everything: spawns the JNI bridge, sets up the env, runs
the policy, prints a results block. `evaluate` plays each game once as
P0 and once as P1, so `--nb_games 1` means 2 games total.

Expected outcome: ~100% WR (UECD-Best vs random bot is a one-sided match).

In [ ]:
import subprocess

result = subprocess.run(
    [
        "microrts-agent",
        "evaluate",
        "--agent",
        str(agent_dir),
        "--opponent",
        "RandomBiasedAI",
        "--maps",
        "maps/open_competition/basesWorkers16x16A.xml",
        "--nb_games",
        "1",
        "--max-steps",
        "2000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 3. Play 1 game vs `CoacAI`

`CoacAI` is a much stronger scripted opponent. Expected outcome:
UECD-Best still wins ~95% on this map (see `data/tournaments/single_map/`
for the full 5-iteration figures).

In [ ]:
result = subprocess.run(
    [
        "microrts-agent",
        "evaluate",
        "--agent",
        str(agent_dir),
        "--opponent",
        "CoacAI",
        "--maps",
        "maps/open_competition/basesWorkers16x16A.xml",
        "--nb_games",
        "1",
        "--max-steps",
        "4000",
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 4. Try other shipped agents

Same flow works on any directory under `data/agents/`. Swap the `--agent`
path for e.g. `data/agents/GridNet-SingleMap` (the baseline), or
`data/agents/UECD-MultiMap-Best` (multi-map agent). All available agents
are listed in [`00_navigate.ipynb`](00_navigate.ipynb) section 5.

## Next steps

- [`02_train.ipynb`](02_train.ipynb): train your own agent from scratch.
- [`03_tournament.ipynb`](03_tournament.ipynb): pit several agents against each other.
- For the full thesis-scale results (19-agent round-robin), see [`data/tournaments/single_map/`](../data/tournaments/single_map/).
- For game-recording mp4s of UECD-Best vs every opponent, see [`data/recordings/`](../data/recordings/).